# MicroCT Ingestion Tutorial

End-to-end walkthrough of the **new MicroCT ingestion pipeline** added in
commit `7f84110`. You will:

1. See how the file parsers (`quantdb.extract_microct`) turn cassava-hosted
   CSV / GraphML / JSON into flat dicts.
2. Run the orchestrator (`quantdb.ingest_microct`) against a synthetic
   payload — ingest, round-trip, idempotency, delete.
3. Drop down to the lower-level generic API (`quantdb.generic_ingest.Ingest`)
   for bringing your own dataset.

**Audience**: developers who already know the quantdb schema (instances →
values → descriptors with provenance) and SQLAlchemy 2.0. This tutorial does
not re-explain the schema.

**Pipeline at a glance**

```
cassava (curation-export.json + path-metadata.json + raw files)
   │
   ▼  classify_path_metadata_files() routes by file type
   │
   ▼  parse_*_morphology() / parse_fascicle_graphml() / parse_data_wrapper()
   │  (returns flat dicts with STRING FK labels)
   │
   ▼  merged into one data_dicts dict
   │
   ▼  ingest_microct(session, models, data_dicts)
   │     - resolves string labels → integer FK ids
   │     - inserts in FK-safe parent-first order
   │     - disables USER triggers around bulk inserts
   ▼
PostgreSQL: quantdb schema
```


## Prereqs

| What | Where |
|------|-------|
| Local Postgres on `localhost:5432` | trust auth for `postgres` user |
| Database `quantdb_test` restored from production dump | see `test/conftest.py:rebuild_database` |
| `sql/inserts_microct.sql` applied | seeds the MicroCT-specific descriptor labels |
| Python env with `quantdb` installed | `pip install -e .` |

Sections 0–1 (parser tour) run **without** a DB — they use offline string
fixtures lifted from `test/test_extract_microct.py`. Sections 2–4 are
DB-bound and self-skip via a `DB_OK` guard if the DB is unreachable.

The dataset UUID we exercise throughout:

```python
MICROCT_UUID = 'fb1cbd05-4320-4d8b-ac3a-44f1fe810718'  # REVA CD MicroCT
```


In [ ]:
import io
import json
from pprint import pprint

from sqlalchemy import create_engine, event, text
from sqlalchemy.orm import Session

from quantdb.utils import dbUri
from quantdb.models import reflect_models
import quantdb.extract_microct as ex
import quantdb.ingest_microct as ig
import quantdb.generic_ingest as gi

print('extract_microct  ->', ex.__file__)
print('ingest_microct   ->', ig.__file__)
print('generic_ingest   ->', gi.__file__)
print('MICROCT_UUID     ->', ex.MICROCT_UUID)
print('PIXEL_TO_UM      ->', ex.PIXEL_TO_UM)
print('PIXEL_TO_UM2     ->', ex.PIXEL_TO_UM2)


## 1. Parser tour (offline)

Every parser in `quantdb.extract_microct` returns a **flat dict** with the
same four keys (some parsers omit keys they don't produce):

```python
{
    'values_inst':     [ {dataset, id_formal, type, desc_inst, id_sub, id_sam}, ... ],
    'instance_parent': [ {'id': {dataset, id_formal}, 'parent': {dataset, id_formal}}, ... ],
    'values_quant':    [ {value, value_blob, object, desc_inst, desc_quant, instance: {...}}, ... ],
    'values_cat':      [ {value_open, value_controlled, object, desc_inst, desc_cat, instance: {...}}, ... ],
}
```

Two non-obvious rules:

1. **FK fields hold string labels, not integer PKs.** `desc_inst='nerve'`,
   `desc_quant='nerve cross section diameter um'`, `value_controlled='true'`.
   `ingest_microct` resolves them at insert time.
2. **Pixel→um conversion happens inside the parser.** Never re-multiply
   downstream. `PIXEL_TO_UM = 11.4`, `PIXEL_TO_UM2 = 129.96`.


In [ ]:
# NerveMorphology CSV — pixel units, converted to um/um2 inside the parser.
SAMPLE_NERVE_CSV = (
    'index,area,perimeter,eq_diameter,center_x,center_y,'
    'major_axis,minor_axis,angle\n'
    '0,100,50.0,11.28,256.5,128.3,15.2,10.1,45.0\n'
    '1,,,,,,,,\n'
    '2,200,75.5,16.0,260.1,130.5,20.3,14.7,30.5\n'
)

nerve_out = ex.parse_nerve_morphology(
    SAMPLE_NERVE_CSV,
    object_uuid='aaaaaaaa-1111-2222-3333-444444444401',
    dataset_uuid=ex.MICROCT_UUID,
    nerve_id_formal='nerve-SR042-CL1-trunk',
    id_sub='sub-SR042',
    id_sam='sam-SR042-CL1',
)

print('keys:', list(nerve_out))
print('values_inst rows :', len(nerve_out['values_inst']))
print('values_quant rows:', len(nerve_out['values_quant']))
print()
print('First instance:')
pprint(nerve_out['values_inst'][0])
print('First quant row (note value already in um):')
pprint(nerve_out['values_quant'][0])


In [ ]:
# RawFascicleTracking GraphML — nodes -> fascicle instances + quant rows,
# edges -> categorical "true"/"false" rows for identity / split / merge.
SAMPLE_GRAPHML = '''<?xml version="1.0" encoding="UTF-8"?>
<graphml xmlns="http://graphml.graphdrawing.org/xmlns">
  <key id="d0" for="node" attr.name="area" attr.type="int"/>
  <key id="d1" for="node" attr.name="equivalent_diameter" attr.type="double"/>
  <key id="d2" for="node" attr.name="centroid-0" attr.type="double"/>
  <key id="d3" for="node" attr.name="centroid-1" attr.type="double"/>
  <key id="d4" for="node" attr.name="ellipse_major_axis" attr.type="double"/>
  <key id="d5" for="node" attr.name="ellipse_minor_axis" attr.type="double"/>
  <key id="d6" for="node" attr.name="ellipse_angle" attr.type="double"/>
  <key id="d7" for="node" attr.name="frame" attr.type="int"/>
  <key id="d8" for="edge" attr.name="identity" attr.type="boolean"/>
  <key id="d9" for="edge" attr.name="split" attr.type="boolean"/>
  <key id="d10" for="edge" attr.name="merge" attr.type="boolean"/>
  <graph id="G" edgedefault="directed">
    <node id="0">
      <data key="d0">50</data><data key="d1">7.98</data>
      <data key="d2">100.5</data><data key="d3">200.3</data>
      <data key="d4">12.0</data><data key="d5">8.0</data>
      <data key="d6">25.0</data><data key="d7">0</data>
    </node>
    <node id="1">
      <data key="d0">60</data><data key="d1">8.74</data>
      <data key="d2">110.2</data><data key="d3">205.1</data>
      <data key="d4">13.5</data><data key="d5">9.2</data>
      <data key="d6">30.0</data><data key="d7">1</data>
    </node>
    <edge source="0" target="1">
      <data key="d8">true</data><data key="d9">false</data><data key="d10">false</data>
    </edge>
  </graph>
</graphml>
'''

graphml_out = ex.parse_fascicle_graphml(
    SAMPLE_GRAPHML,
    object_uuid='aaaaaaaa-1111-2222-3333-444444444402',
    dataset_uuid=ex.MICROCT_UUID,
    nerve_id_formal='nerve-SR042-CL1-trunk',
    id_sub='sub-SR042',
    id_sam='sam-SR042-CL1',
)

print('keys              :', list(graphml_out))
print('fascicle instances:', len(graphml_out['values_inst']))
print('quant rows        :', len(graphml_out['values_quant']))
print('cat rows (edges)  :', len(graphml_out['values_cat']))
print()
print('Fascicle id_formal pattern:')
for vi in graphml_out['values_inst']:
    print(' ', vi['id_formal'])
print()
print('Sample edge cat row:')
pprint(graphml_out['values_cat'][0])


In [ ]:
# SummaryMorphology BranchMorph CSV — values already in mm/mm2, no conversion.
SAMPLE_BRANCH_CSV = (
    'subject,sample,branch_name,interlex_id,'
    'median_nerve_diam_mm,sd_nerve_diam_mm,'
    'median_nerve_area_mm2,sd_nerve_area_mm2,endo_area_mm2,num_fas,'
    'avg_fas_diam_mm,sd_fas_diam_mm,min_fas_diam_mm,max_fas_diam_mm,'
    'measurement_dist_mm,measurement_frame,target\n'
    'SR042,CL1,left_cervical_cardiac_branch,ILX:12345,'
    '0.5,0.1,0.196,0.05,0.12,3,0.15,0.03,0.1,0.2,5.0,42,cardiac\n'
)

summary_out = ex.parse_summary_morphology(
    SAMPLE_BRANCH_CSV,
    object_uuid='aaaaaaaa-1111-2222-3333-444444444403',
    dataset_uuid=ex.MICROCT_UUID,
)

print('values_inst rows :', len(summary_out['values_inst']))
print('values_quant rows:', len(summary_out['values_quant']))
print()
print('Summary instance (id_formal is synthesized from subject+sample+name):')
pprint(summary_out['values_inst'][0])
print('First quant row (mm — no pixel multiplier applied):')
pprint(summary_out['values_quant'][0])


In [ ]:
# Trunk parsers — per-slice files in SummaryMorphology/ directory with
# pixel units AND a dist_global column (in mm).
SAMPLE_TRUNK_NERVE_CSV = (
    'index,segment,dist_global,'
    'area,perimeter,eq_diameter,center_x,center_y,'
    'major_axis,minor_axis,angle\n'
    '0,seg-A,0.0,150,60.0,13.82,260,140,18,12,40\n'
    '1,seg-A,0.5,160,62.0,14.27,261,141,18.5,12.3,41\n'
)

trunk_nerve = ex.parse_trunk_nerve_morphology(
    SAMPLE_TRUNK_NERVE_CSV,
    object_uuid='aaaaaaaa-1111-2222-3333-444444444404',
    dataset_uuid=ex.MICROCT_UUID,
    trunk_id_formal='trunk-SR042-left_cervical_trunk',
    id_sub='sub-SR042',
    id_sam='sam-SR042-CL1',
)

SAMPLE_TRUNK_FAS_CSV = (
    'index,segment,dist_global,'
    'area,equivalent_diameter,centroid-0,centroid-1,'
    'ellipse_major_axis,ellipse_minor_axis,ellipse_angle\n'
    '0,seg-A,0.0,40,7.14,100,200,11,7,20\n'
    '0,seg-A,0.0,45,7.57,105,205,12,8,22\n'
)

trunk_fas = ex.parse_trunk_fas_morphology(
    SAMPLE_TRUNK_FAS_CSV,
    object_uuid='aaaaaaaa-1111-2222-3333-444444444405',
    dataset_uuid=ex.MICROCT_UUID,
    trunk_id_formal='trunk-SR042-left_cervical_trunk',
    id_sub='sub-SR042',
    id_sam='sam-SR042-CL1',
)

print('trunk nerve  -> instances/quant:',
      len(trunk_nerve['values_inst']), len(trunk_nerve['values_quant']))
print('trunk fascic -> instances/quant:',
      len(trunk_fas['values_inst']), len(trunk_fas['values_quant']))
print()
print('Trunk fascicle id_formal carries slice + row index:')
for vi in trunk_fas['values_inst']:
    print(' ', vi['id_formal'])


In [ ]:
# DataWrapper JSON — imaging metadata only; does NOT feed any value table.
# Take it as a string; the parser does json.loads internally.
SAMPLE_WRAPPER_JSON = json.dumps({
    'description': {
        'modality': 'microct',
        'subject_id': 'SR042',
        'sample_id': 'CL1',
    },
    'pixel_properties': {
        'dim': 3,
        'size_x': 2048, 'size_y': 2048, 'size_z': 5000,
        'physical_size_x': 11.4,
        'physical_size_y': 11.4,
        'physical_size_z': 11.4,
    },
})

wrapper = ex.parse_data_wrapper(
    SAMPLE_WRAPPER_JSON,
    object_uuid='aaaaaaaa-1111-2222-3333-444444444406',
)
pprint(wrapper)


In [ ]:
# classify_path_metadata_files routes raw path-metadata entries to the
# correct parser by suffix. In a real run you'd pass cassava's
# path-metadata.json; here we hand-roll a few entries.
fake_pm = {
    'data': [
        {
            'basename': 'SR042-CL1-left_cervical_trunk-NerveMorphology.csv',
            'dataset_relative_path':
                'derivative/sub-SR042/NerveMorphology/'
                'SR042-VagalTrunks/SR042-left_cervical_trunk/'
                'SR042-CL1-left_cervical_trunk-NerveMorphology.csv',
            'remote_id': 'package:abcd-0001',
        },
        {
            'basename': 'SR042-CL1-left_cervical_trunk-RawFascicleTracking.graphml',
            'dataset_relative_path':
                'derivative/sub-SR042/FascicleMorphology/'
                'SR042-VagalTrunks/SR042-left_cervical_trunk/'
                'SR042-CL1-left_cervical_trunk-RawFascicleTracking.graphml',
            'remote_id': 'package:abcd-0002',
        },
        {
            'basename': 'SR042-left-BranchMorph.csv',
            'dataset_relative_path':
                'derivative/sub-SR042/SummaryMorphology/SR042-left-BranchMorph.csv',
            'remote_id': 'package:abcd-0003',
        },
        {
            'basename': 'SR042-CL1-MicroCTWrapper.json',
            'dataset_relative_path':
                'primary/sub-SR042/sam-SR042-CL1/SR042-CL1-MicroCTWrapper.json',
            'remote_id': 'package:abcd-0004',
        },
    ],
}

routed = ex.classify_path_metadata_files(fake_pm)
for bucket, entries in routed.items():
    print(f'{bucket:20s} -> {len(entries)} entry')


**Recap.** Parsers do three things:

1. Mint child instances (slices, fascicles) under the parent nerve id.
2. Emit one quant row per `(instance, descriptor)` pair, already in target units.
3. (GraphML only) emit cat rows for boolean edge properties as
   `value_controlled='true'/'false'`.

A real pipeline run would: (a) `fetch_cassava_metadata` → `extract_microct_objects`
→ `extract_microct_entities` for the dataset/sample skeleton, (b)
`classify_path_metadata_files` to route each file, (c) call the right parser
per file, (d) **merge** all the lists into one `data_dicts` payload, (e)
hand it to `ingest_microct`.

The next sections do step (d) and (e) with a hand-built payload so you
don't need network access.


## 2. Connect to `quantdb_test`

Everything below this point needs a live DB. We attach a `connect` event
listener that runs `SET search_path TO quantdb, public` on every new
connection — **without it `reflect_models` finds the wrong tables and the
ingester silently misbehaves.**

The connection check is wrapped in `try/except`. The rest of the notebook
short-circuits via the `DB_OK` flag so cells render cleanly even when the
DB is down.


In [ ]:
engine = create_engine(
    dbUri(
        dbuser='postgres',
        host='localhost',
        port=5432,
        database='quantdb_test',
    ),
)


@event.listens_for(engine, 'connect')
def _set_search_path(dbapi_connection, connection_record):
    cursor = dbapi_connection.cursor()
    cursor.execute('SET search_path TO quantdb, public')
    cursor.close()


print('engine ->', engine.url)


In [ ]:
try:
    models = reflect_models(engine=engine)
    DB_OK = True
    print('reflected', len(models.Base.classes), 'classes from', engine.url.database)
except Exception as e:
    models = None
    DB_OK = False
    print('DB offline ->', type(e).__name__, '-', e)

DB_OK


## 3. Ingest, round-trip, idempotency, delete

Build a tiny in-process payload (1 subject, 2 samples, 2 nerves, 2 slices,
3 fascicles, plus the corresponding quant + cat rows). This mirrors
`_build_synthetic_data()` in `test/test_ingest_microct.py:74-391` but
trimmed for readability.

A few schema landmines that will bite you when hand-crafting payloads:

- `id_sub` must match `^sub-` and `id_sam` must match `^sam-` (Postgres
  CHECK constraints).
- `type='subject'` rows require `id_sub == id_formal` and `id_sam IS NULL`.
- `type='below'` rows must carry at least one of `id_sub` / `id_sam`.
- All `desc_inst` / `desc_quant` / `desc_cat` labels must already exist —
  they're seeded by `sql/inserts_microct.sql`. Missing labels surface as
  `gi.LookupTableError` from the ingester.


In [ ]:
DATASET = ex.MICROCT_UUID
OBJ1 = 'aaaaaaaa-1111-2222-3333-444444444401'
OBJ2 = 'aaaaaaaa-1111-2222-3333-444444444402'


def _ip(child, parent):
    return {
        'id':     {'dataset': DATASET, 'id_formal': child},
        'parent': {'dataset': DATASET, 'id_formal': parent},
    }


def build_synthetic():
    objects = [
        {'id': DATASET, 'id_type': 'dataset', 'id_file': None},
        {'id': OBJ1,    'id_type': 'package', 'id_file': 100001},
        {'id': OBJ2,    'id_type': 'package', 'id_file': 100002},
    ]
    dataset_object = [
        {'dataset': DATASET, 'object': OBJ1},
        {'dataset': DATASET, 'object': OBJ2},
    ]

    values_inst = [
        # subject — id_sub == id_formal, no id_sam
        {'dataset': DATASET, 'id_formal': 'sub-SR042', 'type': 'subject',
         'desc_inst': 'human', 'id_sub': 'sub-SR042'},
        # samples
        {'dataset': DATASET, 'id_formal': 'sam-SR042-CL1', 'type': 'sample',
         'desc_inst': 'tissue', 'id_sub': 'sub-SR042', 'id_sam': 'sam-SR042-CL1'},
        {'dataset': DATASET, 'id_formal': 'sam-SR042-CL2', 'type': 'sample',
         'desc_inst': 'tissue', 'id_sub': 'sub-SR042', 'id_sam': 'sam-SR042-CL2'},
        # nerves
        {'dataset': DATASET, 'id_formal': 'nerve-CL1-trunk', 'type': 'below',
         'desc_inst': 'nerve', 'id_sub': 'sub-SR042', 'id_sam': 'sam-SR042-CL1'},
        {'dataset': DATASET, 'id_formal': 'nerve-CL2-branch', 'type': 'below',
         'desc_inst': 'nerve', 'id_sub': 'sub-SR042', 'id_sam': 'sam-SR042-CL2'},
        # slices
        {'dataset': DATASET, 'id_formal': 'nerve-CL1-trunk-slice-0', 'type': 'below',
         'desc_inst': 'nerve-cross-section',
         'id_sub': 'sub-SR042', 'id_sam': 'sam-SR042-CL1'},
        {'dataset': DATASET, 'id_formal': 'nerve-CL2-branch-slice-0', 'type': 'below',
         'desc_inst': 'nerve-cross-section',
         'id_sub': 'sub-SR042', 'id_sam': 'sam-SR042-CL2'},
        # fascicles (all attached to slice-0 of the trunk)
        *[
            {'dataset': DATASET, 'id_formal': f'nerve-CL1-trunk-frame-0-fasc-{n}',
             'type': 'below', 'desc_inst': 'fascicle-cross-section',
             'id_sub': 'sub-SR042', 'id_sam': 'sam-SR042-CL1'}
            for n in (1, 2, 3)
        ],
    ]

    instance_parent = [
        _ip('sam-SR042-CL1', 'sub-SR042'),
        _ip('sam-SR042-CL2', 'sub-SR042'),
        _ip('nerve-CL1-trunk', 'sam-SR042-CL1'),
        _ip('nerve-CL2-branch', 'sam-SR042-CL2'),
        _ip('nerve-CL1-trunk-slice-0', 'nerve-CL1-trunk'),
        _ip('nerve-CL2-branch-slice-0', 'nerve-CL2-branch'),
        *[_ip(f'nerve-CL1-trunk-frame-0-fasc-{n}',
              'nerve-CL1-trunk-slice-0')
          for n in (1, 2, 3)],
    ]

    nerve_descs = [
        'nerve cross section area um2',
        'nerve cross section diameter um',
    ]
    fasc_descs = [
        'fascicle cross section area um2',
        'fascicle cross section diameter um',
    ]

    values_quant = []
    v = 1.0
    for slice_id, obj in [
        ('nerve-CL1-trunk-slice-0',  OBJ1),
        ('nerve-CL2-branch-slice-0', OBJ1),
    ]:
        for desc in nerve_descs:
            values_quant.append({
                'value': v, 'value_blob': v,
                'object': obj,
                'desc_inst': 'nerve-cross-section',
                'desc_quant': desc,
                'instance': {'dataset': DATASET, 'id_formal': slice_id},
            })
            v += 1
    for n in (1, 2, 3):
        fasc_id = f'nerve-CL1-trunk-frame-0-fasc-{n}'
        for desc in fasc_descs:
            values_quant.append({
                'value': v, 'value_blob': v,
                'object': OBJ2,
                'desc_inst': 'fascicle-cross-section',
                'desc_quant': desc,
                'instance': {'dataset': DATASET, 'id_formal': fasc_id},
            })
            v += 1

    values_cat = [
        {'value_controlled': 'true', 'value_open': None,
         'object': OBJ2, 'desc_inst': 'fascicle-cross-section',
         'desc_cat': 'fascicleEdgeIdentity',
         'instance': {'dataset': DATASET,
                      'id_formal': f'nerve-CL1-trunk-frame-0-fasc-{n}'}}
        for n in (1, 2, 3)
    ]

    return {
        'objects': objects,
        'dataset_object': dataset_object,
        'values_inst': values_inst,
        'instance_parent': instance_parent,
        'values_quant': values_quant,
        'values_cat': values_cat,
    }


data_dicts = build_synthetic()
{k: len(v) for k, v in data_dicts.items()}


### Insert order

`ingest_microct` (`quantdb/ingest_microct.py:394-617`) inserts in this
order — every step satisfies a prerequisite of the next:

```
1. objects                # PKs other rows reference
2. dataset_object         # registers the dataset's object set
3. values_inst            # the things being measured
4. instance_parent        # hierarchy (needs values_inst PKs)
5. obj_desc_inst          # required by trigger check_desc_inst_exists
6. obj_desc_quant         # required by trigger values_quant_check_before
7. obj_desc_cat           # required by trigger values_cat_check_before
8. values_quant           # leaf data
9. values_cat             # leaf data
```

Around each bulk insert the orchestrator runs
`ALTER TABLE ... DISABLE TRIGGER USER` so per-row trigger checks (which
are heavy and would fire `O(N)` times) get skipped, then re-enables.
**`USER` not `ALL`** — `ALL` is denied on AWS RDS.


In [ ]:
if not DB_OK:
    print('skipping ingest — no DB')
else:
    s = models.Session()
    try:
        before = ig._count_microct(s, models)
        ig.ingest_microct(s, models, data_dicts)
        s.commit()
        after = ig._count_microct(s, models)
        delta = {k: after[k] - before[k] for k in after}

        print('before MicroCT counts:', before)
        print('after  MicroCT counts:', after)
        print('delta                :', delta)
    except Exception:
        s.rollback()
        raise
    finally:
        s.close()


> `_count_microct` is a private helper (underscore prefix). It's stable
> enough for tutorial diagnostics, but don't depend on it from production
> code — use a `select(func.count())` against the relevant table.


In [ ]:
# Round-trip read: pull the data back out as flat dicts and compare to the
# in-memory payload.
if not DB_OK:
    print('skipping round-trip — no DB')
else:
    s = models.Session()
    try:
        extracted = ig.extract_microct_from_db(s, models)

        print('extracted keys ->', list(extracted))
        print('counts:')
        for k, v in extracted.items():
            print(f'  {k:18s} {len(v):>4d}')

        print()
        print('First extracted quant row (note string FK labels — same shape '
              'as cell that built the payload):')
        pprint(extracted['values_quant'][0])
    finally:
        s.close()


In [ ]:
# Idempotency: re-ingest the SAME payload and counts must not change. The
# orchestrator relies on (object, instance, desc_quant) and
# (object, instance, desc_cat) UNIQUE constraints + ON CONFLICT DO NOTHING
# on obj_desc_*.
if not DB_OK:
    print('skipping idempotency — no DB')
else:
    s = models.Session()
    try:
        before = ig._count_microct(s, models)
        ig.ingest_microct(s, models, data_dicts)
        s.commit()
        after = ig._count_microct(s, models)
        delta = {k: after[k] - before[k] for k in after}

        print('idempotency delta (must be all zero):', delta)
        assert all(d == 0 for d in delta.values()), 'NOT idempotent'
    except Exception:
        s.rollback()
        raise
    finally:
        s.close()


In [ ]:
# Delete: removes ONLY MicroCT rows in FK-safe child-first order.
if not DB_OK:
    print('skipping delete — no DB')
else:
    from sqlalchemy import func, select
    from quantdb.ingest_v2 import F006_UUID

    s = models.Session()
    try:
        # Sanity-snapshot a non-MicroCT dataset's count so we can prove we
        # didn't touch it.
        VI = models.ValuesInst
        f006_before = s.execute(
            select(func.count()).select_from(VI).where(VI.dataset == F006_UUID)
        ).scalar_one()

        ig.delete_microct_data(s, models)
        s.commit()

        after = ig._count_microct(s, models)
        f006_after = s.execute(
            select(func.count()).select_from(VI).where(VI.dataset == F006_UUID)
        ).scalar_one()

        print('MicroCT counts after delete (all zero):', after)
        print(f'f006 values_inst before/after delete: '
              f'{f006_before} / {f006_after}  (untouched)')
    except Exception:
        s.rollback()
        raise
    finally:
        s.close()


## 4. Bring your own dataset — `Ingest`

The MicroCT pipeline is built on top of `quantdb.generic_ingest.Ingest`,
which is what you reach for when you have a one-off table to populate or
a non-MicroCT dataset to ingest.

Cheat sheet:

| Method                                | Purpose                                   |
|---------------------------------------|-------------------------------------------|
| `Ingest(models)`                      | Construct (one arg — session is per-call) |
| `ing.row(s, table, **cols)`           | Single row, with FK label resolution      |
| `ing.batch(s, table, [rows])`         | Many rows, shared FK cache                |
| `ing.get(s, table, **cols)`           | Lookup, returns ORM instance or `None`    |
| `ing.session()`                       | Context manager: commit on exit, rollback on error |

Errors:

- `gi.LookupTableError` — FK label doesn't exist in a lookup table
  (e.g. you passed `desc_quant='nonexistent'`).
- `gi.IngestError` — a downstream `IntegrityError` / trigger violation.


In [ ]:
if not DB_OK:
    print('skipping Ingest demo — no DB')
else:
    ing = gi.Ingest(models)
    s = models.Session()
    try:
        # Lookup an existing unit by label.
        um_row = ing.get(s, 'units', label='um')
        print('units(label=um) ->', um_row)
        print('  id =', um_row.id, ' label =', um_row.label)

        # Lookup something that doesn't exist — returns None, NOT an error.
        missing = ing.get(s, 'units', label='this-unit-does-not-exist')
        print('missing unit    ->', missing)
    finally:
        s.close()


In [ ]:
# Single-row insert with auto FK resolution. Use a fresh session and roll
# back at the end so this cell is repeatable.
if not DB_OK:
    print('skipping row/batch demo — no DB')
else:
    ing = gi.Ingest(models)
    s = models.Session()
    try:
        # The dataset object must exist before we can attach instances to it.
        # build_synthetic() already inserted it; re-issuing is a no-op.
        ing.row(s, 'objects',
                id=ex.MICROCT_UUID, id_type='dataset', id_file=None)

        # Single instance — note string label desc_inst='human', resolved
        # automatically.
        try:
            ing.row(s, 'values_inst',
                    dataset=ex.MICROCT_UUID,
                    id_formal='sub-DEMO',
                    type='subject',
                    desc_inst='human',
                    id_sub='sub-DEMO')
            print('inserted sub-DEMO')
        except gi.IngestError as e:
            # Likely already exists from a previous run.
            print('skip (already exists):', e)

        # Batch insert with a shared FK cache — much faster than N row()
        # calls when many rows share the same descriptor labels.
        sample_rows = [
            {'dataset': ex.MICROCT_UUID,
             'id_formal': f'sam-DEMO-{i}',
             'type': 'sample',
             'desc_inst': 'tissue',
             'id_sub': 'sub-DEMO',
             'id_sam': f'sam-DEMO-{i}'}
            for i in range(3)
        ]
        try:
            ing.batch(s, 'values_inst', sample_rows)
            print('inserted 3 samples via batch()')
        except gi.IngestError as e:
            print('skip (already exists):', e)

    finally:
        s.rollback()  # demo cell — keep the DB clean
        s.close()


## Pointers

- **Add a new file-type parser** → mirror `parse_nerve_morphology` in
  `quantdb/extract_microct.py:315`. Return the same four-key dict shape.
- **New dataset orchestrator** → mirror `ingest_microct` in
  `quantdb/ingest_microct.py:394`. Reuse `Ingest.batch()` for small tables;
  drop to bulk SQL with `DISABLE TRIGGER USER` for `values_inst`,
  `values_quant`, `values_cat`.
- **Synthetic-payload template** → `_build_synthetic_data` in
  `test/test_ingest_microct.py:74`.
- **Pulling cassava data** → `ex.fetch_cassava_metadata(uuid)` returns
  `(curation_export, path_metadata)`; combine with
  `extract_microct_objects` and `extract_microct_entities` to seed the
  skeleton, then `classify_path_metadata_files` + the parsers for the
  measurements.
- **Lower-level FK resolution** → see `quantdb/generic_ingest.py:deep_upsert`
  if you want to understand what `Ingest` is doing under the hood.
